# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

#질문하는 일련의 과정을 하나의 함수로 묶어두었음
def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [2]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "세종대왕의 이름은",
    "Q: 세종대왕의 이름은 무엇인가요? A:",
    "세종대왕의 실명은",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

프롬프트: 세종대왕의 이름은
생성: 세종대왕의 이름은 이도(李祹)입니다.

프롬프트: Q: 세종대왕의 이름은 무엇인가요? A:
생성: 세종대왕의 이름은 이도(李祹)입니다.

프롬프트: 세종대왕의 실명은
생성: 세종대왕의 실명은 이도(李祹)입니다.



---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [3]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

=== Zero-shot ===
첫 번째 리뷰는 부정적인 요소가 포함되어 있어 부정으로 판정할 수 있습니다. 두 번째 리뷰는 긍정적인 내용이므로 긍정으로 판정할 수 있습니다.


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → 긍정 | 6/10  
입력: "최고의 영화, 또 보고 싶다" → 긍정 | 10/10


In [7]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 괜찮은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

괜찮은 : 불안한


In [8]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 괜찮은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

Few-shot 프롬프트:
다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 괜찮은 :

생성된 완성:
좋지 않은


---
## 2. 역할(Role) 설정 

In [9]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮다고 여겨집니다. 많은 연구에 따르면, 하루 400mg 이하의 카페인은 대부분의 사람들에게 안전하다고 합니다. 이는 대략 커피 4잔에 해당합니다. 그러나 개인의 카페인 민감도, 건강 상태, 임신 여부 등에 따라 다를 수 있으므로, 자신의 몸 상태를 잘 살펴보는 것이 중요합니다.

카페인을 과다 섭취하면 불안, 불면증, 심장 두근거림 등의 부작용이 나타날 수 있으니, 이러한 증상이 나타난다면 섭취량을 줄이는 것이 좋습니다. 또한, 특정 건강 문제가 있는 경우에는 의사와 상담하는 것이 바람직합니다.




In [ ]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

=== 영양사 역할 ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮습니다. 평균적으로 한 잔의 커피에는 약 95mg의 카페인이 포함되어 있습니다. 따라서 하루 4잔을 마신다면 약 380mg의 카페인을 섭취하게 됩니다.

미국 식품의약국(FDA)에서는 건강한 성인이 하루에 최대 400mg의 카페인을 섭취하는 것을 안전하다고 보고하고 있습니다. 그러나 개인의 카페인 민감도는 다를 수 있으므로, 불안감, 불면증, 심장 두근거림 등의 증상이 나타난다면 섭취량을 줄이는 것이 좋습니다.

또한, 커피를 마실 때는 추가된 설탕이나 크림의 양도 고려해야 합니다. 건강한 식습관을 유지하기 위해서는 균형 잡힌 식사를 하고, 충분한 수분을 섭취하는 것이 중요합니다.


In [17]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 쓰레기 역할 ===')
print(ask(
    '''You are a blunt, sarcastic character.
    Speak rudely and dismissively in Korean.
    Still answer the question, but with a harsh tone.''',
    question,
    temperature=0.8,
))

=== 쓰레기 역할 ===
괜찮을지 안 괜찮을지 뭘 그렇게 고민해? 커피 마시고 싶으면 마시고, 몸이 힘들면 그때 가서 생각해봐. 하루에 4잔 정도면 그리 큰 문제는 아닐 거야. 어차피 많은 사람들이 그렇게 살잖아. 그냥 더 마시고 싶으면 마시고, 나중에 후회하는 건 네 자유니까.


In [14]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 문제 해결, 이해 등을 수행할 수 있도록 하는 기술입니다. AI는 주로 머신러닝과 딥러닝 기술을 활용하여 데이터를 분석하고, 패턴을 인식하며, 예측을 수행합니다. 

일상생활에서 AI는 음성 인식, 이미지 처리, 자율주행차, 추천 시스템 등 다양한 분야에서 활용되고 있습니다. 인공지능은 더 효율적인 작업 수행과 의사결정을 지원하며, 인간의 일상생활을 더욱 편리하게 만들어주는 역할을 하고 있습니다. 

더 궁금한 점이 있으시면 언제든지 말씀해 주세요!

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(Artificial Intelligence, AI)은 기계가 인간의 지능을 모방하거나 인간처럼 사고하고 행동할 수 있도록 하는 컴퓨터 과학의 한 분야입니다. AI는 기계 학습(Machine Learning), 자연어 처리(Natural Language Processing), 컴퓨터 비전(Computer Vision), 로봇 공학(Robotics) 등 여러 하위 분야로 나뉩니다.

기계 학습은 데이터에서 패턴을 학습하여 예측하거나 결정을 내리는 알고리즘을 개발하는 데 중점을 둡니다. 이는 통계학과 최적화 기법을 활용하여 모델을 훈련시키고, 새로운 데이터에 대해 일반화할 수 있도록 합니다.

자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성할 수 있도록 하는 기술로, 번역, 감정 분석, 챗봇 개발 등 다양한 응용 분야에서 사용됩니다. 

컴퓨터 비전은 이미지와 비디오에서 정보를 추출하고 해석하는 기술로, 얼굴 인식, 객체 감지, 자율주행차의 시각적 인식 등에서 활용됩니다.

AI는 의료, 금융, 제조, 소매 등 여러 산업에서 혁신적인 변화를 가져오고 있으며, 데이터 분석, 자동화, 고객 서비스 개선 등 다양한 분야에 응용되고 있습니다. 그러나 AI 기술의 발전과 함께 윤리적 문제, 개인정보 보호, 일자리 대체 등

---
## 3. 출력 형식 지정 

In [ ]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))  #프롬프트 뒤에 형식을 덧붙이는 식으로
print('\n' + '=' * 50 + '\n')

=== Markdown ===
### 재택근무의 장단점

#### 장점
- **유연한 근무 시간**: 개인의 생활 패턴에 맞춰 근무 시간을 조정할 수 있어 일과 삶의 균형을 맞추기 용이하다.
- **통근 시간 절약**: 출퇴근 시간을 없애거나 줄일 수 있어 더 많은 시간을 개인적인 활동이나 업무에 할애할 수 있다.
- **비용 절감**: 교통비, 식비 등 출근에 드는 비용을 줄일 수 있으며, 기업 측면에서도 사무실 유지 비용을 절감할 수 있다.

#### 단점
- **사회적 고립감**: 동료와의 직접적인 소통이 줄어들어 외로움을 느낄 수 있으며, 팀워크가 약화될 수 있다.
- **업무 집중력 저하**: 가정에서의 다양한 방해 요소로 인해 업무에 집중하기 어려울 수 있다.
- **경계 모호**: 일과 개인 생활의 경계가 흐려져 업무 시간이 늘어날 수 있으며, 이는 스트레스를 유발할 수 있다.




In [16]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목       | 장점                          | 단점                          |
|------------|-------------------------------|-------------------------------|
| 유연성     | 근무 시간과 장소의 유연성    | 일과 개인 생활의 경계 모호함  |
| 교통비 절감 | 출퇴근 시간과 비용 절감      | 사회적 상호작용 부족          |
| 생산성     | 집중할 수 있는 환경 조성     | 자율성 부족 시 생산성 저하    |


---
### 출력 형식 (표 / JSON)

In [18]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

=== Table ===
| 요인         | 설명                                   | 대응                       |
|--------------|--------------------------------------|--------------------------|
| 화학물질의 농도 | 식각 공정에 사용되는 화학물질의 농도가 수율에 영향을 미침 | 농도 최적화 및 모니터링   |
| 식각 시간     | 식각 시간이 너무 짧거나 길면 원하는 패턴이 제대로 형성되지 않음 | 시간 조정 및 테스트      |
| 온도         | 공정 온도가 적절하지 않으면 식각 속도가 변동하여 수율에 영향 | 온도 제어 및 안정화      |


In [19]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

=== JSON ===
{
  "factors": [
    {
      "name": "공정 파라미터",
      "action": "식각 속도, 온도, 압력 등의 최적화가 필요"
    },
    {
      "name": "마스크 품질",
      "action": "마스크의 정밀도와 결함이 수율에 직접적인 영향을 미침"
    },
    {
      "name": "화학물질 순도",
      "action": "사용되는 화학물질의 순도가 낮으면 불순물로 인해 결함 발생"
    }
  ]
}

파싱 성공: ['factors']


---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [20]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

=== CoT 없음 ===
전체 25명 중 40%는 10명입니다. 현재 A팀은 10명으로, 전체 인원의 40%에 해당합니다. 따라서 A팀에 추가 배치가 필요하지 않습니다. A팀의 인원이 40%를 초과하지 않기 때문에 추가 인원 배치가 필요하지 않습니다.


In [ ]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',  #일명 '마법의 문장'
))

=== CoT 적용 ===
전체 인원은 25명입니다. A팀, B팀, C팀의 인원 수는 다음과 같습니다:

- A팀: 10명
- B팀: 15명
- C팀: 5명

1. **전체 인원 중 A팀의 비율 계산하기**:
   A팀의 인원 수는 10명입니다. 전체 인원 25명 중 A팀의 비율을 계산해 보겠습니다.

   \[
   A팀 비율 = \frac{A팀 인원}{전체 인원} = \frac{10}{25} = 0.4
   \]

2. **비율을 퍼센트로 변환하기**:
   위의 계산 결과를 퍼센트로 변환하면:

   \[
   A팀 비율 = 0.4 \times 100 = 40\%
   \]

3. **40% 이상인지 확인하기**:
   문제에서 요구하는 것은 A팀의 인원이 전체 인원의 40% 이상이어야 한다는 것입니다. 현재 A팀의 비율은 정확히 40%입니다.

4. **결론 도출하기**:
   40% 이상이라는 조건을 만족하기 위해서는 A팀의 비율이 40%를 초과해야 합니다. 현재 A팀의 비율은 40%이므로, 추가 인원 배치가 필요하지 않습니다.

따라서, **A팀에 추가 배치가 필요하지 않습니다.**


In [22]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3,000원


In [23]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

먼저, 연필과 지우개의 가격을 계산해보겠습니다.

1. 연필의 가격:
   - 연필 1자루의 가격은 500원입니다.
   - 연필 5자루의 가격은 500원 × 5 = 2500원입니다.

2. 지우개의 가격:
   - 지우개 1개의 가격은 300원입니다.
   - 지우개 3개의 가격은 300원 × 3 = 900원입니다.

3. 총 금액 계산:
   - 연필의 총 가격 2500원과 지우개의 총 가격 900원을 더합니다.
   - 2500원 + 900원 = 3400원입니다.

따라서, 총 금액은 3400원입니다.

답: 3400원


In [24]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3900원


In [25]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

단계별 풀이:
1. 연필 5자루의 가격: 5 × 500 = 2500원
2. 지우개 3개의 가격: 3 × 300 = 900원
3. 총 금액: 2500 + 900 = 3400원

답: 3400원


In [26]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 논리적으로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

먼저 연필과 지우개의 가격을 계산해보겠습니다.

1. 연필의 가격:
   - 연필 1자루의 가격은 500원입니다.
   - 연필 5자루의 가격은 500원 × 5 = 2500원입니다.

2. 지우개의 가격:
   - 지우개 1개의 가격은 300원입니다.
   - 지우개 3개의 가격은 300원 × 3 = 900원입니다.

이제 두 금액을 합산해보겠습니다.

총 금액 = 연필 가격 + 지우개 가격
총 금액 = 2500원 + 900원 = 3400원

따라서, 총 금액은 3400원입니다.

답: 3400원


---
## 5. System + User 프롬프트 

In [27]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

결론: 목요일 회의에 맞춰 우선순위를 조정합니다.

- 월요일: 기획안 제출 마감 (변경 없음)
- 수요일: 코드 리뷰를 목요일로 연기
- 금요일: 발표 준비 및 발표 진행 (변경 없음)


In [28]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

| 결론                                      | 리스크                                      | 가정                                         |
|-----------------------------------------|------------------------------------------|--------------------------------------------|
| VIP 이탈 방지를 위한 개인 맞춤형 재접근 프로그램 개발 및 실행 | 고객의 반응이 예상보다 저조할 수 있음          | 고객 데이터 분석이 정확하고 신뢰할 수 있다.      |
| VIP 고객 전용 이벤트 및 특별 할인 제공 | 이벤트나 할인 혜택이 고객의 이탈을 막지 못할 수 있음 | 고객이 이벤트 및 혜택에 관심을 가질 것이라고 가정. |
| VIP 고객과의 개인적인 소통 강화        | 소통이 고객에게 부담으로 작용할 수 있음         | 개인화된 소통이 고객의 재구매로 이어질 것이라고 가정. |
| 고객의 피드백 수집 및 분석              | 피드백 수집이 고객의 불만을 초래할 수 있음       | 고객이 피드백에 응답할 것이라고 가정.          |
| 장기적인 관계 구축을 위한 로열티 프로그램 강화 | 로열티 프로그램이 고객의 이탈을 막지 못할 수 있음 | 고객이 로열티 프로그램의 혜택을 인지할 것이라고 가정. |

위 액션 플랜을 통해 VIP 고객의 이탈을 최소화하고, 장기적인 관계를 구축하기 위한 노력을 기울일 수 있습니다.


---
## 6. Self-check Prompting

In [ ]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)

---
## 7. Constraint Prompting

In [ ]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)